In [3]:
import tkinter as tk
import transformers
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---------------- LOAD MODEL ----------------
model_path = r"C:\Users\lakshita\Desktop\Adv ML sem 6\new emotional_detection"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

# ---------------- LABELS ----------------
label_names = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization",
    "relief","remorse","sadness","surprise","neutral"
]

# ---------------- EMOTION MAP ----------------
emotion_map = {
    "joy": ("happy 😊", "#FFD700"),
    "love": ("happy ❤️", "#FF69B4"),
    "sadness": ("sad 😢", "#6495ED"),
    "anger": ("angry 😡", "#FF4500"),
    "fear": ("fear 😨", "#9370DB"),
    "surprise": ("surprise 😲", "#00CED1"),
    "neutral": ("neutral 😐", "#333333")
}

# ---------------- TEXT PREPROCESSING ----------------
SLANG = {
    "u": "you", "ur": "your", "gr8": "great",
    "luv": "love", "im": "i am",
    "dont": "do not", "cant": "cannot", "wont": "will not"
}

NEGATIONS = {"not", "no", "never", "dont", "cannot", "can't", "won't"}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)

    words = text.split()
    words = [SLANG.get(w, w) for w in words]

    # negation handling
    result = []
    negate = False
    window = 0

    for w in words:
        if w in NEGATIONS:
            negate = True
            window = 3

        if negate and window > 0:
            result.append("NOT_" + w)
            window -= 1
        else:
            result.append(w)

        if window == 0:
            negate = False

    return " ".join(result)

# ---------------- REASON ENGINE ----------------
def generate_reason(text, emotion, confidence):
    words = set(text.split())

    if confidence < 0.45:
        return "Sentence is ambiguous, mixed emotions"

    if "NOT_" in text:
        return "Negation detected which affects emotion"

    if emotion.startswith("happy"):
        return "Positive words detected but sentence is uncertain"

    if emotion.startswith("sad"):
        return "Negative emotional words detected"

    return "Contextual emotional pattern detected"

# ---------------- PREDICTION ----------------
def predict(text):
    text = preprocess(text)

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    top2 = torch.topk(probs, 2)

    idx1 = top2.indices[0][0].item()
    idx2 = top2.indices[0][1].item()
    conf = top2.values[0][0].item()

    fine1 = label_names[idx1]
    fine2 = label_names[idx2]

    emotion, color = emotion_map.get(fine1, (fine1, "#000000"))
    alt = emotion_map.get(fine2, (fine2,))[0]

    reason = generate_reason(text, emotion, conf)

    return emotion, conf, alt, reason, color

# ---------------- GUI ----------------
root = tk.Tk()
root.title("Emotion Detection AI")
root.geometry("600x450")
root.config(bg="#eef3ff")

frame = tk.Frame(root, bg="white")
frame.place(relx=0.5, rely=0.5, anchor="center", width=520, height=400)

tk.Label(frame, text="Emotion Detection System",
         font=("Arial", 18, "bold"), bg="white").pack(pady=10)

text_input = tk.Text(frame, height=5, width=50)
text_input.pack(pady=10)

result_label = tk.Label(frame, text="", font=("Arial", 12), bg="white", justify="left")
result_label.pack(pady=10)

def on_predict():
    text = text_input.get("1.0", tk.END)

    emotion, conf, alt, reason, color = predict(text)

    output = f"""
Emotion: {alt}
Confidence: {int(conf*100)}%
Alternate: {emotion}
Reason: {reason}
"""

    result_label.config(text=output, fg=color)

tk.Button(frame, text="Detect Emotion",
          bg="#4a6cff", fg="white",
          command=on_predict).pack(pady=5)

root.mainloop()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]